In [33]:
import requests
import pandas as pd
import time
import json

def fetch_huge_hh_data(query='аналитик данных OR data analyst OR junior analyst OR стажер аналитик', max_pages=20):
    url = 'https://api.hh.ru/vacancies'
    all_items = []
    
    for page in range(max_pages):
        params = {
            'text': query, 
            'area': 113, 
            'per_page': 100, 
            'page': page  # Вот тут магия: меняем номер страницы в цикле
        }
        
        try:
            response = requests.get(url, params=params)
            # Проверка, что сервер не "послал" нас (код 200 = успех)
            if response.status_code == 200:
                page_data = response.json()
                items = page_data.get('items', [])
                
                if not items:
                    print(f"На странице {page} вакансий больше нет. Стоп.")
                    break
                
                all_items.extend(items)
                print(f"Загружена страница {page}. Всего собрано: {len(all_items)}")
                
                # Запоминаем общее количество найденного (только на первой итерации)
                if page == 0:
                    print(f"Всего на сайте найдено по запросу: {page_data.get('found')}")
            else:
                print(f"Ошибка {response.status_code} на странице {page}")
                break
                
        except Exception as e:
            print(f"Произошла ошибка: {e}")
            break
            
        # ВАЖНО: делаем паузу, чтобы HH не забанил за подозрительную активность
        time.sleep(0.4) 
        
    return all_items

# 1. сбор данных (20 страниц по 100 = до 2000 вакансий)
raw_vacancies_list = fetch_huge_hh_data()

# 2. Сохраняем СЫРОЙ JSON (для истории и Гитхаба)
with open('hh_huge_raw_data.json', 'w', encoding='utf-8') as f:
    json.dump(raw_vacancies_list, f, ensure_ascii=False, indent=4)

# 3. сразу перегоняю в пандас для анализа
df = pd.json_normalize(raw_vacancies_list)
print(f"Количество строк в готовой таблице: {len(df)}")

Загружена страница 0. Всего собрано: 100
Всего на сайте найдено по запросу: 12844
Загружена страница 1. Всего собрано: 200
Загружена страница 2. Всего собрано: 300
Загружена страница 3. Всего собрано: 400
Загружена страница 4. Всего собрано: 500
Загружена страница 5. Всего собрано: 600
Загружена страница 6. Всего собрано: 700
Загружена страница 7. Всего собрано: 800
Загружена страница 8. Всего собрано: 900
Загружена страница 9. Всего собрано: 1000
Загружена страница 10. Всего собрано: 1100
Загружена страница 11. Всего собрано: 1200
Загружена страница 12. Всего собрано: 1300
Загружена страница 13. Всего собрано: 1400
Загружена страница 14. Всего собрано: 1500
Загружена страница 15. Всего собрано: 1600
Загружена страница 16. Всего собрано: 1700
Загружена страница 17. Всего собрано: 1800
Загружена страница 18. Всего собрано: 1900
Загружена страница 19. Всего собрано: 2000
Количество строк в готовой таблице: 2000


In [44]:
df[['url', 'name']]

,url,name
0,https://api.hh.ru/vacancies/131034550?host=hh.ru,Специалист по документообороту / Помощник отде...
1,https://api.hh.ru/vacancies/131019221?host=hh.ru,Руководитель по бережливому производству
2,https://api.hh.ru/vacancies/130971766?host=hh.ru,Аналитик данных (SQL)
3,https://api.hh.ru/vacancies/131044588?host=hh.ru,Бизнес-аналитик/Аналитик данных
4,https://api.hh.ru/vacancies/130557118?host=hh.ru,Junior product manager
...,...,...
1995,https://api.hh.ru/vacancies/130747286?host=hh.ru,Менеджер маркетплейса ОZON
1996,https://api.hh.ru/vacancies/130861708?host=hh.ru,Менеджер по работе с маркетплейсами
1997,https://api.hh.ru/vacancies/130992981?host=hh.ru,Ассистент менеджера по закупкам
1998,https://api.hh.ru/vacancies/130355540?host=hh.ru,Системный аналитик


In [ ]:
df.groupby('url')

In [46]:
# 1. Создаем фильтр по названию (чтобы убрать юристов и менеджеров)
# Ищем аналитиков, но исключаем мусор
mask_name = df['name'].str.contains('аналитик|analyst|data|junior|стажер', case=False, na=False)
mask_trash = df['name'].str.contains('юрист|hr|менеджер маркетплейс|продаж|директор', case=False, na=False)

# 2. Создаем фильтр по SQL (ищем в требованиях)
# Мы заглядываем в колонку snippet.requirement, где обычно пишут стек
mask_sql = df['snippet.requirement'].str.contains('SQL', case=False, na=False)

# 3. Применяем всё вместе
# Знак & означает "и то, и другое", знак ~ означает "не"
df_target = df[mask_name & ~mask_trash & mask_sql].copy()

# 4. Оставляем только те колонки, которые ты просила
result = df_target[['name', 'alternate_url']]

print(f"🔥 Найдено {len(result)} вакансий, где нужен SQL и это точно аналитика!")
result.head(100) # Выведет первые 20 ссылок

🔥 Найдено 151 вакансий, где нужен SQL и это точно аналитика!


,name,alternate_url
14,Системный / Бизнес-аналитик,https://hh.ru/vacancy/130913571
18,Junior Data Analyst,https://hh.ru/vacancy/130980230
30,Младший аналитик данных,https://hh.ru/vacancy/130899253
36,Системный аналитик,https://hh.ru/vacancy/130902238
56,Младший аналитик данных,https://hh.ru/vacancy/131012665
...,...,...
1351,Data Engineer/Data Quality (офис),https://hh.ru/vacancy/131024440
1365,Аналитик данных в службу внутреннего аудита,https://hh.ru/vacancy/130739878
1366,Аналитик A/B платформы,https://hh.ru/vacancy/130898651
1374,Аналитик данных (Junior),https://hh.ru/vacancy/130587067
